# Feature Engineering

## 1. Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np
import datetime as dt

# Load dataset
df = pd.read_csv('../data/online_retail.csv')

# Quick clean
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Avoid Target Leakage by Splitting Observation vs Future Timeline
cutoff_date = df['InvoiceDate'].quantile(0.70)
obs_df = df[df['InvoiceDate'] <= cutoff_date]
target_df = df[df['InvoiceDate'] > cutoff_date]
print(f"Observation window ends at: {cutoff_date}")

Observation window ends at: 2011-10-09 10:23:36


## 2. Preprocessing

import datetime as dt

# Calculate RFM explicitly ONLY on Observation Window
snapshot_date = obs_df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = obs_df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})
rfm.rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalPrice': 'MonetaryValue'}, inplace=True)
rfm.head()

In [2]:
def aggregate_user_journey(user_df):
    user_df = user_df.sort_values('InvoiceDate')
    first_touch = user_df.iloc[0]
    last_touch = user_df.iloc[-1]
    features = {
        'CustomerID': first_touch['CustomerID'],
        'first_country': first_touch['Country'],
        'time_to_conversion_days': (last_touch['InvoiceDate'] - first_touch['InvoiceDate']).days
    }
    return pd.Series(features)

print('Aggregating auxiliary journey features from strictly Observation Window...')
journey_features = obs_df.groupby('CustomerID').apply(aggregate_user_journey).reset_index(drop=True)

Aggregating auxiliary journey features from strictly Observation Window...


C:\Users\priya\AppData\Local\Temp\ipykernel_5540\2170216354.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  journey_features = obs_df.groupby('CustomerID').apply(aggregate_user_journey).reset_index(drop=True)


from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. K-Means Segmentation on historical RFM profile
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)
kmeans = KMeans(n_clusters=3, random_state=42)
rfm['Segment'] = kmeans.fit_predict(rfm_scaled)

# Merge RFM with journey features
final_df = pd.merge(rfm, journey_features, on='CustomerID', how='inner')

# 2. Extract TRUE FUTURE value from explicitly holdout target_df 
future_revenue = target_df.groupby('CustomerID')['TotalPrice'].sum().reset_index()
future_revenue.columns = ['CustomerID', 'Future_Value']

# Map future value back to our observation users
final_df = pd.merge(final_df, future_revenue, on='CustomerID', how='left')
final_df['Future_Value'] = final_df['Future_Value'].fillna(0) # 0 means churned / no future value observed

# Create rigorous classification target: Did they purchase again in the future window?
final_df['converted'] = (final_df['Future_Value'] > 0).astype(int)
final_df = final_df.drop(columns=['Future_Value'])

# One-hot encode categorical channels
final_df = pd.get_dummies(final_df, columns=['first_country'], drop_first=True)

final_df.to_csv('../data/processed_data.csv', index=False)
print('Dataset finalized with Strict Forward Time-Splitting and Demographic Segmentation.')
display(final_df.head(3))

Let's perform the primary RFM (Recency, Frequency, Monetary) computations below to define the baseline dataset value.

## 5. RFM Calculation
Recency, Frequency, Monetary value per CustomerID.

In [3]:
# Calculate RFM explicitly ONLY on Observation Window
snapshot_date = obs_df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = obs_df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})
rfm.rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalPrice': 'MonetaryValue'}, inplace=True)
rfm.head()

,Recency,Frequency,MonetaryValue
CustomerID,,,
12346.0,265,1,77183.60
12347.0,69,5,2790.86
12348.0,14,4,1797.24
12350.0,249,1,334.40
12352.0,11,7,2194.31


## 6. Save Processed Dataset

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. K-Means Segmentation on historical RFM profile
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)
kmeans = KMeans(n_clusters=3, random_state=42)
rfm['Segment'] = kmeans.fit_predict(rfm_scaled)

# Merge RFM with journey features
final_df = pd.merge(rfm, journey_features, on='CustomerID', how='inner')

# 2. Extract TRUE FUTURE value from explicitly holdout target_df 
future_revenue = target_df.groupby('CustomerID')['TotalPrice'].sum().reset_index()
future_revenue.columns = ['CustomerID', 'Future_Value']

# Map future value back to our observation users
final_df = pd.merge(final_df, future_revenue, on='CustomerID', how='left')
final_df['Future_Value'] = final_df['Future_Value'].fillna(0) # 0 means churned

# Create vigorous classification target: Did they purchase again in the future window?
final_df['converted'] = (final_df['Future_Value'] > 0).astype(int)
final_df = final_df.drop(columns=['Future_Value'])

# One-hot encode categorical channels
final_df = pd.get_dummies(final_df, columns=['first_country'], drop_first=True)

final_df.to_csv('../data/processed_data.csv', index=False)
print('Dataset finalized with Strict Forward Time-Splitting and Demographic Segmentation.')
display(final_df.head(3))

C:\Users\priya\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\priya\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\priya\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\priya\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 966, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\priya\AppData\Local\Programs\Python\Python310\lib\subp

Dataset finalized with Strict Forward Time-Splitting and Demographic Segmentation.


,CustomerID,Recency,Frequency,MonetaryValue,Segment,time_to_conversion_days,converted,first_country_Austria,first_country_Bahrain,first_country_Belgium,...,first_country_Portugal,first_country_Saudi Arabia,first_country_Singapore,first_country_Spain,first_country_Sweden,first_country_Switzerland,first_country_USA,first_country_United Arab Emirates,first_country_United Kingdom,first_country_Unspecified
0,12346.0,265,1,77183.60,2,0,0,False,False,False,...,False,False,False,False,False,False,False,False,True,False
1,12347.0,69,5,2790.86,0,237,1,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,12348.0,14,4,1797.24,0,282,0,False,False,False,...,False,False,False,False,False,False,False,False,False,False
